# HF_TB Model

## Imports

In [2]:
import LLMs, TP2
from LLMs import *
from TP2 import Dataset

C:\Users\Luco1421\Desktop\U\ia\TPs\TP2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Luco1421\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Charge model

In [3]:
LLMs.clean_gpu()

hugging_face_tb = LLMs.charge_model("HuggingFaceTB/SmolLM3-3B")

Loading weights: 100%|██████████| 326/326 [00:00<00:00, 7380.31it/s]
C:\Users\Luco1421\Desktop\U\ia\TPs\TP2\.venv\Lib\site-packages\torch\nn\modules\module.py:1370: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:39.)
  return t.to(


## Function to request HF_TB's answer

In [4]:
def chat_hf_tb(text: str, prompt: str):
    messages = [
        {"role": "system", "content": prompt},
        {"role": "user", "content": text},
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,
            temperature=0.1,
            top_p=0.1,
            do_sample=True
        )

    response_text = tokenizer.decode(outputs[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)

    LLMs.clean_gpu()

    return extract_results(response_text)

### Example of use

In [6]:
tokenizer, model = hugging_face_tb
chat_hf_tb(SAMPLE_1 + SAMPLE_2 + SAMPLE_3, CONTEXT + TEXT_BEGINNER)

[1, 0, 1]

# Results

## Load dataset and test utils class

In [7]:
dataset = TP2.Dataset("FEINA_1.xlsx").read()
llm_utils = LLMs.TestLLMUtils()

# Test without Few Shots

In [12]:
llm_utils.test_without_shots("Hugging Face TB", chat_hf_tb, dataset)

Hugging Face tb - Accuracy: 0.4913690239439981


# Test with Few Shots

In [18]:
llm_utils.test_with_shots("Hugging Face TB", chat_hf_tb, dataset, [2, 4, 7])

Hugging Face TB with 2 shots - Accuracy: 0.49706518381217174
Hugging Face TB with 4 shots - Accuracy: 0.4851085310449268
Hugging Face TB with 7 shots - Accuracy: 0.48815086782376504


In [10]:
llm_utils.test_all_LLM("Hugging Face TB", chat_hf_tb, dataset, [2])

Hugging Face TB without shots: average = 0.5500, std = 0.0707
Hugging Face TB with 2 shots - average = 0.5000, std = 0.1414
--------------------------------------------------


{0: [0.6, 0.5], 2: [0.6, 0.4]}

In [3]:
LLMs.clean_gpu()

del hugging_face_tb, chat_hf_tb, tokenizer, model
gc.collect()

NameError: name 'hugging_face_tb' is not defined